# Building a Custom Evaluation Metric in scikit-learn


scikit-learn provides many common metrics for classification and regression, but real projects sometimes need a score that better matches a specific business goal. For example, you might want to combine several metrics, add custom penalties, or reward one type of correct prediction more than another.

When that happens, `make_scorer` lets you wrap a Python function so that it can be used with tools such as cross-validation, grid search, and model comparison workflows.


The workflow is straightforward:

```mermaid
flowchart LR
    A[Decide what good<br>performance means] --> B[Write a metric<br>function]
    B --> C[Wrap it with<br>make_scorer]
    C --> D[Use it in cross-validation<br>or model selection]
```


## Set Up a Classification Example


We reuse the water-quality dataset from the previous classification notebooks. The target column `is_safe` indicates whether a water sample is safe to drink.


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import cross_validate

In [ ]:
df = pd.read_csv("data/water-quality.csv")
df.head()

As before, it is helpful to inspect the target distribution. The dataset is imbalanced, so relying on a single metric such as accuracy would hide useful information about model quality.


In [ ]:
df["is_safe"].value_counts()

Suppose we care about three things at once:

- **precision**, because false positives are costly,
- **recall**, because missed safe samples are also costly,
- **F1 score**, because it balances the two.

One simple way to reflect that priority is to average the three metrics into a single custom score.


In [ ]:
X = df.drop(columns="is_safe")
y = df["is_safe"]

model = LogisticRegression(max_iter=1000)

## Define the Custom Metric


A scorer function receives the true labels and the predicted labels. Here, we compute precision, recall, and F1 score, then return their average.

We use `zero_division=0` so the function remains stable even if a model predicts no positive samples in one fold.


In [ ]:
def custom_scorer(y_true, y_pred):
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return (precision + recall + f1) / 3

`make_scorer` converts the plain Python function into a scorer object that scikit-learn can use. If you ever build a loss where smaller values are better, you can also pass `greater_is_better=False`.


In [ ]:
custom_metric = make_scorer(custom_scorer)
custom_metric

## Evaluate with Cross-Validation


To see how the custom score behaves, we evaluate the model with five-fold cross-validation. We also keep the built-in precision, recall, and F1 scorers so we can compare them side by side.


In [ ]:
scoring = {
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "custom_score": custom_metric,
}

cv_results = cross_validate(model, X, y, cv=5, scoring=scoring)

In [ ]:
pd.DataFrame(
    {
        "metric": ["precision", "recall", "f1", "custom_score"],
        "mean_cv_score": [
            cv_results["test_precision"].mean(),
            cv_results["test_recall"].mean(),
            cv_results["test_f1"].mean(),
            cv_results["test_custom_score"].mean(),
        ],
        "std_cv_score": [
            cv_results["test_precision"].std(),
            cv_results["test_recall"].std(),
            cv_results["test_f1"].std(),
            cv_results["test_custom_score"].std(),
        ],
    }
)

The custom score sits on the same `0` to `1` scale as the component metrics because it is just their average. That makes it easy to compare models with one number while still remembering how the score was constructed.


In [ ]:
custom_scores = pd.Series(cv_results["test_custom_score"], name="custom_score")
print(f"Average custom score: {custom_scores.mean():.3f}")
custom_scores

Sometimes the business problem values one metric more than the others. For example, if missing positive cases is especially costly, you might design a scorer that gives recall a larger weight than precision.

In [ ]:
def recall_focused_scorer(y_true, y_pred):
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    return (precision + 2 * recall + f1) / 4


recall_focused_metric = make_scorer(recall_focused_scorer)
weighted_results = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring={"custom_score": custom_metric, "recall_focused": recall_focused_metric},
)

pd.DataFrame(
    {
        "metric": ["custom_score", "recall_focused"],
        "mean_cv_score": [
            weighted_results["test_custom_score"].mean(),
            weighted_results["test_recall_focused"].mean(),
        ],
    }
)

Custom scorers are especially useful when standard metrics only capture part of what you care about. As long as you can express your evaluation rule as a Python function, `make_scorer` lets you plug it into the rest of the scikit-learn ecosystem.
